# 🤖 Affiliate Video Bot v7 — 2026

**AI tạo video affiliate viral cho TikTok & Shopee — 10 ngành hàng**

| Ngành hàng | Commission |
|---|---|
| 💆 Beauty/Skincare | 10–15% |
| 💊 Health/Supplement | 12–20% |
| 👗 Fashion | 7–12% |
| 🏠 Home & Living | 8–12% |
| 🐾 Pet Products | 8–15% |
| 👶 Baby/Kids | 8–15% |
| 📱 Tech Accessories | 5–8% |
| 💪 Sports | 7–10% |

---
## ⚡ Quy trình chạy
```
Cell 0 → Điền thông tin (BẮT BUỘC)
Cell 1 → Cài deps (1 lần duy nhất, ~8 phút)
Cell 2 → Mount Drive + Tải fonts
Cell 3 → Tải AI Models (tuỳ chọn)
Cell 4 → 🚀 KHỞI ĐỘNG SERVER (mỗi session)
Cell 5 → Test pipeline
Cell 6 → Tạo video thử
```

> ⚠️ **Bắt buộc: Chọn Runtime → T4 GPU trước khi chạy**

## ⚙️ Cell 0 — Điền Thông Tin Cá Nhân
> **Điền đầy đủ trước khi chạy bất kỳ cell nào khác**

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ⚙️  CELL 0 — ĐIỀN THÔNG TIN (BẮT BUỘC)
# ═══════════════════════════════════════════════════════════════

# 1. GitHub repo chứa code bot v7
GITHUB_REPO = "https://github.com/YOUR_USERNAME/affiliate-video-bot.git"
# ↑ Thay YOUR_USERNAME bằng tên GitHub của bạn

# 2. ngrok authtoken (lấy tại ngrok.com → Dashboard → Authtoken)
NGROK_AUTH_TOKEN = "your_ngrok_authtoken_here"

# 3. URL Render service của bạn
RENDER_URL = "https://affiliate-video-bot-v7.onrender.com"
# ↑ Thay bằng URL thật sau khi deploy Render

# 4. Secret key (phải khớp với COLAB_SECRET trên Render)
BOT_SECRET = "affiliatebot_v7_secret"

# 5. Telegram User ID của bạn
YOUR_TELEGRAM_ID = 0
# ↑ Nhắn @userinfobot để lấy ID → điền số vào đây

# ───────────────────────────────────────────────────────────────
print('✅ Config OK!')
print(f'   GitHub: {GITHUB_REPO[:50]}...')
print(f'   Render: {RENDER_URL}')
print(f'   Secret: {BOT_SECRET}')
print(f'   Telegram ID: {YOUR_TELEGRAM_ID}')
print()
if 'YOUR_USERNAME' in GITHUB_REPO:
    print('⚠️  Chưa điền GITHUB_REPO!')
if YOUR_TELEGRAM_ID == 0:
    print('⚠️  Chưa điền YOUR_TELEGRAM_ID!')
if 'your_ngrok' in NGROK_AUTH_TOKEN:
    print('⚠️  Chưa điền NGROK_AUTH_TOKEN!')
else:
    print('→ Chạy Cell 1 tiếp theo ✓')

## 📦 Cell 1 — Cài Đặt Dependencies
> Chạy **1 lần duy nhất** (~8-10 phút). Session sau bỏ qua nếu đã cài.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📦 CELL 1 — CÀI DEPENDENCIES (1 lần duy nhất)
# ═══════════════════════════════════════════════════════════════

import subprocess, sys

print('📦 Bước 1/4: Cài ffmpeg...')
subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg', 'libsm6', 'libxext6'], check=True)
print('✅ ffmpeg OK')

print('\n📦 Bước 2/4: Cài Python packages cơ bản...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'python-telegram-bot==21.6',
    'flask', 'requests', 'pyngrok',
    'Pillow', 'moviepy', 'imageio', 'imageio-ffmpeg',
    'numpy', 'tqdm',
], check=True)
print('✅ Base packages OK')

print('\n📦 Bước 3/4: Cài AI/ML packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers>=0.30.0',
    'transformers>=4.44.0',
    'accelerate>=0.34.0',
    'safetensors>=0.4.4',
    'open-clip-torch>=2.26.1',
    'huggingface_hub>=0.24.6',
    'xformers',
], check=True)
print('✅ AI packages OK')

print('\n📦 Bước 4/4: Cài AudioCraft (nhạc AI)...')
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'audiocraft'], check=True)
    print('✅ AudioCraft OK')
except Exception as e:
    print(f'⚠️  AudioCraft skip: {e} (nhạc AI không có, dùng Pixabay thay thế)')

print('\n📥 Clone code từ GitHub...')
import os
if not os.path.exists('/content/affiliate-video-bot'):
    subprocess.run(['git', 'clone', GITHUB_REPO, '/content/affiliate-video-bot'], check=True)
    print('✅ Clone OK')
else:
    subprocess.run(['git', '-C', '/content/affiliate-video-bot', 'pull'], check=True)
    print('✅ Pull latest OK')

sys.path.insert(0, '/content/affiliate-video-bot')
os.chdir('/content/affiliate-video-bot')

print('\n' + '='*50)
print('✅ TẤT CẢ CÀI ĐẶT HOÀN TẤT!')
print('→ Chạy Cell 2 tiếp theo')
print('='*50)

## 🔗 Cell 2 — Mount Google Drive + Tải Fonts
> Chạy **1 lần** để setup Drive. Session sau nếu Drive đã mount thì bỏ qua.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🔗 CELL 2 — MOUNT GOOGLE DRIVE + SETUP
# ═══════════════════════════════════════════════════════════════

import sys, json
sys.path.insert(0, '/content/affiliate-video-bot')

from pipeline.drive_manager import setup_drive

print('🔗 Mount Google Drive...')
drive = setup_drive()

print('\n📊 Drive stats:')
stats = drive.drive_stats()
for folder, info in stats.items():
    print(f'   📁 {folder}/: {info["size_mb"]} MB | {info["files"]} files')

print('\n🔤 Tải fonts về Drive (lần đầu ~30 giây)...')
fonts = ['Montserrat-Bold.ttf', 'Montserrat-ExtraBold.ttf', 'BeVietnamPro-Bold.ttf']
for font in fonts:
    path = drive.get_font_path(font)
    status = f'✅ {path}' if path else '⚠️ Không tải được'
    print(f'   {font}: {status}')

print('\n' + '='*50)
print('✅ DRIVE SẴN SÀNG!')
print(f'   Root: {drive.drive_root}')
print('→ Chạy Cell 3 (tuỳ chọn) hoặc thẳng Cell 4')
print('='*50)

## 🤖 Cell 3 — Tải AI Models (Tuỳ Chọn)
> Chỉ cần tải **1 lần** — lưu vào Drive. Session sau load từ Drive, không tải lại.
> **Bỏ qua Cell này** nếu không muốn dùng AI video (dùng MoviePy fallback thay thế).

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🤖 CELL 3 — TẢI AI MODELS (tuỳ chọn, lần đầu ~15-30 phút)
# ═══════════════════════════════════════════════════════════════

import sys
sys.path.insert(0, '/content/affiliate-video-bot')
from pipeline.drive_manager import setup_drive

drive = setup_drive()

print('🤖 Chọn model muốn tải (uncomment dòng tương ứng):')
print()

# ── Model 1: Wan2.1-I2V (TỐT NHẤT) ────────────────────────────
# Chất lượng cinematic, Image-to-Video, cần 12GB VRAM, ~25GB disk
# drive.download_model('Wan-AI/Wan2.1-I2V-14B-480P', 'wan2.1-i2v-14B-480P')

# ── Model 2: CogVideoX-5B (NHANH HƠN) ─────────────────────────
# Text-to-Video, cần 8GB VRAM, ~18GB disk
# drive.download_model('THUDM/CogVideoX-5b', 'cogvideox-5b')

# ── Model 3: FLUX.1-schnell (ẢNH AI NHANH) ────────────────────
# Sinh ảnh chất lượng cao, 4 steps, ~8GB disk
# drive.download_model('black-forest-labs/FLUX.1-schnell', 'flux1-schnell')

# ── Model 4: AnimateDiff (NHẸ NHẤT) ───────────────────────────
# Cần 6GB VRAM, ~7GB disk
# drive.download_model('guoyww/animatediff-motion-adapter-sdxl-beta', 'animatediff-sdxl')

# ── Model 5: MusicGen (NHẠC AI) ────────────────────────────────
# Sinh nhạc nền AI, ~300MB
# drive.download_model('facebook/musicgen-small', 'musicgen-small')

print('💡 Tip: Colab Free T4 = 15GB VRAM')
print('   → Wan2.1 hoặc CogVideoX đều chạy được')
print()
print('Nếu không tải model → bot dùng MoviePy (không cần GPU)')
print('→ Chạy Cell 4 để khởi động server')

## 🚀 Cell 4 — KHỞI ĐỘNG SERVER
> **Chạy cell này MỖI SESSION Colab.** Tự động:
> - Mount Drive
> - Khởi động Flask server
> - Tạo ngrok tunnel
> - **Tự đăng ký URL về Render** (không cần copy-paste thủ công)
> - Gửi thông báo về Telegram

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🚀 CELL 4 — KHỞI ĐỘNG SERVER (chạy mỗi session)
# ═══════════════════════════════════════════════════════════════

import sys, os, json, threading, time
import requests as req_lib
sys.path.insert(0, '/content/affiliate-video-bot')

# ── Mount Drive ───────────────────────────────────────────────
from pipeline.drive_manager import setup_drive
drive = setup_drive()

# ── GPU Info ─────────────────────────────────────────────────
def get_gpu_info():
    try:
        import torch
        if torch.cuda.is_available():
            name = torch.cuda.get_device_name(0)
            mem  = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
            return f'{name} ({mem}GB VRAM)'
        return 'CPU only (không có GPU)'
    except:
        return 'GPU check failed'

# ── Flask Server ──────────────────────────────────────────────
from flask import Flask, request, jsonify
colab_app = Flask('affiliate_colab_v7')

@colab_app.route('/ping')
def ping():
    return jsonify({'status': 'alive', 'gpu': get_gpu_info(), 'version': '7.0'}), 200

@colab_app.route('/info')
def info():
    return jsonify({'status': 'alive', 'gpu': get_gpu_info(),
                    'drive': drive.drive_stats(), 'version': '7.0'}), 200

@colab_app.route('/generate', methods=['POST'])
def generate():
    data   = request.get_json(silent=True) or {}
    secret = data.get('secret', '')
    if secret != BOT_SECRET:
        return jsonify({'error': 'Unauthorized'}), 403
    threading.Thread(
        target=run_pipeline,
        args=(data.get('user_id'), data.get('name', 'SP'),
              data.get('price', ''), data.get('description', ''),
              data.get('platform', 'tiktok'), data.get('callback_url', '')),
        daemon=True
    ).start()
    return jsonify({'status': 'accepted', 'message': 'Đang xử lý...'}), 200

# ── AI Pipeline ───────────────────────────────────────────────
def run_pipeline(user_id, name, price, description, platform, callback_url):
    print(f'\n📥 Task: {name} | {price} | {platform}')
    try:
        from pipeline.product_analyzer import analyze_product
        from pipeline.emotional_engine import build_emotional_package
        from pipeline.viral_caption import generate_viral_caption
        from pipeline.background import get_main_prompt, get_hook_prompt
        from pipeline.video_engine import generate_video

        print('🔍 Phân tích sản phẩm...')
        analysis  = analyze_product(name, description, price)
        emotional = build_emotional_package(name, analysis.category, analysis.gender, price)
        vc        = generate_viral_caption(name, price, analysis.category,
                                            analysis.gender, emotional, platform)

        caption = vc.tiktok if platform == 'tiktok' else vc.shopee
        value_stack = '✅ Freeship toàn quốc\n✅ Đổi trả 7 ngày\n✅ Giao 2-3 ngày'

        ai_prompt  = get_main_prompt(analysis.category, analysis.gender,
                                      product_name=name)
        hook_prompt= get_hook_prompt(analysis.category, analysis.gender)

        print(f'🎬 Tạo video (engine=auto) — ngành: {analysis.category}...')
        output = generate_video(
            ai_prompt=ai_prompt,
            hook_prompt=hook_prompt,
            product_name=name,
            price=price,
            category=analysis.category,
            music_mood=emotional.emotional_music,
            value_stack=value_stack,
            cta=vc.cta_bio,
            badge=emotional.urgency_line[:40],
            gender=analysis.gender,
            engine='auto',
        )

        if output and output.exists():
            print(f'✅ Video: {output} ({output.stat().st_size/1e6:.1f} MB)')
            send_callback(callback_url, {
                'user_id': user_id, 'status': 'success_drive',
                'video_url': str(output),
                'caption': caption[:900], 'error': ''
            })
            # Hiển thị trong notebook
            from IPython.display import Video, display, Markdown
            display(Markdown(f'### ✅ Video: {name}'))
            display(Video(str(output), width=360))
            print(f'\n📋 Caption ({platform}):')
            print(caption[:400])
            print(f'\n🔬 A/B Hooks:')
            for i, h in enumerate(vc.hook_ab_variants[:3], 1):
                print(f'  Hook {i}: {h}')
        else:
            send_callback(callback_url, {
                'user_id': user_id, 'status': 'error',
                'error': 'Video generation failed', 'caption': '', 'video_url': ''
            })

    except Exception as e:
        import traceback
        print(f'❌ Pipeline error: {e}')
        traceback.print_exc()
        send_callback(callback_url, {
            'user_id': user_id, 'status': 'error',
            'error': str(e)[:200], 'caption': '', 'video_url': ''
        })

def send_callback(url, payload):
    if not url: return
    try:
        req_lib.post(url, json=payload,
                     headers={'X-Bot-Secret': BOT_SECRET}, timeout=15)
        print(f'✅ Callback → {url}')
    except Exception as e:
        print(f'⚠️ Callback fail: {e}')

# ── Start Flask in background ─────────────────────────────────
flask_thread = threading.Thread(
    target=lambda: colab_app.run(host='0.0.0.0', port=5001, use_reloader=False),
    daemon=True
)
flask_thread.start()
time.sleep(2)

# ── Khởi động ngrok ───────────────────────────────────────────
from pyngrok import ngrok, conf as ngrok_conf

if NGROK_AUTH_TOKEN and 'your_ngrok' not in NGROK_AUTH_TOKEN:
    ngrok_conf.get_default().auth_token = NGROK_AUTH_TOKEN
    print('✅ ngrok authenticated')
else:
    print('⚠️ Chưa điền NGROK_AUTH_TOKEN — URL sẽ expire sau 2h')

# Kill tunnel cũ nếu có
try: ngrok.kill()
except: pass
time.sleep(1)

tunnel    = ngrok.connect(5001, bind_tls=True)
ngrok_url = tunnel.public_url

gpu_info = get_gpu_info()

print()
print('='*60)
print('🚀 COLAB SERVER V7 ĐANG CHẠY!')
print('='*60)
print(f'🔗 ngrok URL : {ngrok_url}')
print(f'🖥️  GPU       : {gpu_info}')
print(f'🌐 Render    : {RENDER_URL}')
print('='*60)
print()
print('📱 Nếu TỰ ĐĂNG KÝ thất bại, gửi lệnh này vào Telegram:')
print(f'   /setcolab {ngrok_url}')
print()

# ── Tự đăng ký URL về Render (không cần copy-paste) ──────────
if RENDER_URL and 'onrender.com' in RENDER_URL:
    print('📡 Đang tự đăng ký URL về Render...')
    try:
        resp = req_lib.post(
            f'{RENDER_URL}/colab/seturl',
            json={'url': ngrok_url, 'notify_user_id': YOUR_TELEGRAM_ID},
            headers={'X-Bot-Secret': BOT_SECRET, 'Content-Type': 'application/json'},
            timeout=20
        )
        if resp.status_code == 200:
            print('✅ URL đã tự đăng ký thành công!')
            if YOUR_TELEGRAM_ID != 0:
                print(f'📲 Đã gửi thông báo về Telegram ID: {YOUR_TELEGRAM_ID}')
        else:
            print(f'⚠️ Render trả về {resp.status_code} — gửi /setcolab thủ công')
    except Exception as e:
        print(f'⚠️ Tự đăng ký thất bại: {e}')
        print(f'→ Gửi thủ công: /setcolab {ngrok_url}')
else:
    print('⚠️ RENDER_URL chưa set — không tự đăng ký được')
    print(f'→ Gửi thủ công: /setcolab {ngrok_url}')

print()
print('📌 GIỮ NOTEBOOK NÀY MỞ — đóng tab là Colab ngủ!')
print('💡 Dùng /autocolab on để bot tự ping giữ Colab sống')

## 🧪 Cell 5 — Test Pipeline (Không cần GPU)
> Kiểm tra product analyzer + emotional engine + caption generation.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🧪 CELL 5 — TEST PIPELINE (tuỳ chọn)
# ═══════════════════════════════════════════════════════════════

import sys
sys.path.insert(0, '/content/affiliate-video-bot')

from pipeline.product_analyzer import analyze_product
from pipeline.emotional_engine import build_emotional_package
from pipeline.viral_caption import generate_viral_caption

# ── Test nhiều ngành hàng ─────────────────────────────────────
test_cases = [
    ('Serum Vitamin C 30ml', '350k', 'Serum trắng da niacinamide 10%'),
    ('Váy maxi hoa nhí', '299k', 'Váy nữ lụa mềm tay dài'),
    ('Collagen uống Nhật', '890k', 'Collagen peptide 10000mg Shiseido'),
    ('Đèn decor phòng ngủ', '185k', 'Đèn LED cảm ứng dán tường'),
    ('Bánh mochi nhân kem', '129k', 'Bánh mochi Đài Loan ngon'),
    ('Ốp lưng iPhone 15 Pro', '89k', 'Ốp magsafe trong suốt chống sốc'),
    ('Thức ăn mèo Whiskas', '65k', 'Pate mèo vị cá hồi 400g'),
    ('Bộ bodysuit sơ sinh', '145k', 'Bodysuit cotton 100% bé 0-12 tháng'),
]

print('🧪 KIỂM TRA PIPELINE — 8 ngành hàng\n')
print('='*60)

for name, price, desc in test_cases:
    a  = analyze_product(name, desc, price)
    ep = build_emotional_package(name, a.category, a.gender, price)
    vc = generate_viral_caption(name, price, a.category, a.gender, ep, 'tiktok')

    print(f'📦 {name}')
    print(f'   🏷️  Ngành: {a.category} | 👤 Target: {a.gender}')
    print(f'   💡 Lợi ích: {a.key_benefit}')
    print(f'   🎣 Hook: {ep.hook_curiosity[:70]}')
    print(f'   🎵 Nhạc: {ep.emotional_music}')
    print(f'   ⏰ Urgency: {ep.urgency_line[:60]}')
    print(f'   💬 Comment bait: {vc.comment_bait}')
    print()

print('='*60)
print('✅ TẤT CẢ NGÀNH HÀNG HOẠT ĐỘNG!')
print('→ Chạy Cell 6 để test tạo video thật')

## 🎬 Cell 6 — Tạo Video Thử (Test GPU + Full Pipeline)
> Kiểm tra toàn bộ AI pipeline từ đầu đến cuối.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🎬 CELL 6 — TẠO VIDEO THỬ (test GPU pipeline)
# ═══════════════════════════════════════════════════════════════

import sys
sys.path.insert(0, '/content/affiliate-video-bot')

from pipeline.product_analyzer import analyze_product
from pipeline.emotional_engine import build_emotional_package
from pipeline.viral_caption import generate_viral_caption
from pipeline.background import get_main_prompt, get_hook_prompt
from pipeline.video_engine import generate_video
from IPython.display import Video, display, Markdown

# ── Chọn sản phẩm test ────────────────────────────────────────
TEST_NAME  = 'Serum Vitamin C 30ml'
TEST_PRICE = '350k'
TEST_DESC  = 'Serum trắng da niacinamide 10% Hàn Quốc'
TEST_PLAT  = 'tiktok'

# ── Chạy pipeline ─────────────────────────────────────────────
print(f'🎬 Đang tạo video: {TEST_NAME}')
print('='*50)

print('\n1️⃣ Phân tích sản phẩm...')
analysis  = analyze_product(TEST_NAME, TEST_DESC, TEST_PRICE)
emotional = build_emotional_package(TEST_NAME, analysis.category, analysis.gender, TEST_PRICE)
vc        = generate_viral_caption(TEST_NAME, TEST_PRICE, analysis.category,
                                    analysis.gender, emotional, TEST_PLAT)

print(f'   ✅ Ngành: {analysis.category} | Target: {analysis.gender}')
print(f'   ✅ Nhạc: {emotional.emotional_music}')
print(f'   ✅ Hook: {emotional.hook_curiosity[:60]}')

print('\n2️⃣ Sinh AI prompt...')
ai_prompt   = get_main_prompt(analysis.category, analysis.gender, product_name=TEST_NAME)
hook_prompt = get_hook_prompt(analysis.category, analysis.gender)
print(f'   ✅ Prompt: {ai_prompt[:80]}...')

print('\n3️⃣ Tạo video (engine=auto)...')
print('   Thứ tự thử: Wan2.1 → CogVideoX → AnimateDiff → Slideshow → MoviePy')
print('   (Thời gian: 30 giây - 5 phút tuỳ engine có VRAM)\n')

output = generate_video(
    ai_prompt=ai_prompt,
    hook_prompt=hook_prompt,
    product_name=TEST_NAME,
    price=TEST_PRICE,
    category=analysis.category,
    music_mood=emotional.emotional_music,
    value_stack='✅ Freeship toàn quốc\n✅ Đổi trả 7 ngày\n✅ Dermatologist tested',
    cta=vc.cta_bio,
    badge=emotional.urgency_line[:35],
    gender=analysis.gender,
    engine='auto',
)

print('='*50)
if output and output.exists():
    size_mb = output.stat().st_size / 1_000_000
    print(f'✅ VIDEO HOÀN THÀNH!')
    print(f'   📁 {output}')
    print(f'   📏 {size_mb:.1f} MB')
    display(Markdown('### 🎬 Preview Video:'))
    display(Video(str(output), width=360, height=640))

    print(f'\n📋 CAPTION TIKTOK:')
    print('-'*40)
    print(vc.tiktok[:500])

    print(f'\n🔬 A/B HOOKS (test 3 cái này):')
    for i, h in enumerate(vc.hook_ab_variants[:3], 1):
        print(f'  Hook {i}: {h}')
else:
    print('❌ Video thất bại — kiểm tra log ở trên')
    print('Tip: Thử đổi engine="moviepy" để test không cần GPU')

## 📊 Cell 7 — Thống Kê & Nghiên Cứu Thị Trường 2026
> Dữ liệu affiliate Vietnam Q1 2026 — dùng để chọn ngành hàng tối ưu.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📊 CELL 7 — THỊ TRƯỜNG AFFILIATE VIETNAM 2026
# ═══════════════════════════════════════════════════════════════

market_data = {
    'TikTok Shop Vietnam (H1 2025)': '$3.57 tỷ GMV (+148% YoY)',
    'Thị phần TikTok/e-commerce VN': '42% toàn bộ thị trường',
    'Thị trường affiliate VN 2024': '$700M - $1 tỷ USD',
    'Tăng trưởng 2022-2024': '30x trong 2 năm',
    'Social commerce VN 2026': '$20.98 tỷ USD dự kiến',
}

viral_formula = {
    '0-2s (HOOK)': 'Quyết định 70% retention — Pattern interrupt + Emotion',
    '2-6s (REVEAL)': 'Social proof + Transformation — tạo desire',
    '6-10s (VALUE)': 'Authenticity + Identity — build trust',
    '10-13s (CTA)': 'Urgency + FOMO + Comment bait (×4 reach)',
    '13-15s (LOOP)': 'Seamless loop → tăng completion rate → thuật toán đẩy',
}

commission_2026 = {
    'Health/Supplement': ('12-20%', '⭐⭐⭐⭐⭐', 'Collagen, Vitamin, Protein'),
    'Beauty/Skincare':   ('10-15%', '⭐⭐⭐⭐⭐', 'Serum, Kem dưỡng, Mặt nạ'),
    'Pet Products':      ('8-15%',  '⭐⭐⭐⭐',  'Thức ăn cao cấp, Grooming'),
    'Baby/Kids':         ('8-15%',  '⭐⭐⭐⭐',  'An toàn, Organic, Giáo dục'),
    'Home & Living':     ('8-12%',  '⭐⭐⭐⭐',  'Decor, Dụng cụ nhà bếp'),
    'Fashion':           ('7-12%',  '⭐⭐⭐⭐',  'Streetwear, Áo dài, Swimwear'),
    'Sports':            ('7-10%',  '⭐⭐⭐',   'Gym, Yoga, Running'),
    'Tech Accessories':  ('5-8%',   '⭐⭐⭐',   'Ốp lưng, Tai nghe, Cáp'),
    'Food/Snack':        ('5-10%',  '⭐⭐⭐',   'Snack nhập, Cà phê, Trà'),
}

print('='*60)
print('📊 THỊ TRƯỜNG AFFILIATE VIETNAM 2026')
print('='*60)
for k, v in market_data.items():
    print(f'  {k}: {v}')

print('\n' + '='*60)
print('🎬 CÔNG THỨC VIDEO VIRAL (OpusClip analysis 34,635 clips)')
print('='*60)
for timestamp, formula in viral_formula.items():
    print(f'  {timestamp}: {formula}')

print('\n' + '='*60)
print('💰 COMMISSION RATE 2026 — XẾP HẠNG TỐT NHẤT')
print('='*60)
print(f'{"Ngành hàng":<22} {"Commission":<12} {"Rating":<12} {"Sản phẩm hot"}')
print('-'*70)
for cat, (comm, rating, products) in commission_2026.items():
    print(f'{cat:<22} {comm:<12} {rating:<12} {products}')

print('\n💡 Gợi ý chiến lược 2026:')
print('  1. Health + Beauty = commission cao + CTR cao (transformation rõ rệt)')
print('  2. Comment bait = ×4 reach (thuật toán TikTok ưu tiên comment)')
print('  3. Loop seamless = completion rate cao = thuật toán đẩy nhiều hơn')
print('  4. A/B test 3 hooks = tối ưu CTR trước khi tăng ngân sách quảng cáo')